# Customer Churn Prediction in the Nigerian Telecom Sector

Nigeria's telecommunications industry is one of the fastest-growing in Africa, with over 200 million subscribers across major providers. Despite this growth, churn — the rate at which customers stop using a provider's services — remains a pressing business challenge. Providers like **MTN Nigeria**, **Airtel Nigeria**, **Glo**, and **9mobile** face fierce competition, and retaining existing customers is far more cost-effective than acquiring new ones.

According to industry reports, acquiring a new telecom customer can cost up to five times more than retaining an existing one. A churn rate reduction of even 5% can significantly boost profitability. This makes churn prediction not just an academic exercise but a genuine business priority.

As the data analyst on this project, your goal is to build machine learning models that can predict whether a customer is likely to churn, and determine which model performs better for deployment in a real-world context.

You have been provided with two datasets from a regional telecom analytics firm:

- `telecom_demographics.csv` contains customer demographic information:

| Variable             | Description                                      |
|----------------------|--------------------------------------------------|
| `customer_id`        | Unique identifier for each customer.             |
| `telecom_partner`    | The telecom provider associated with the customer (MTN, Airtel, Glo, 9mobile). |
| `gender`             | The gender of the customer.                      |
| `age`                | The age of the customer.                         |
| `state`              | The Nigerian state in which the customer is located. |
| `city`               | The city in which the customer is located.       |
| `pincode`            | The area code of the customer's location.        |
| `registration_event` | When the customer first registered with the telecom provider. |
| `num_dependents`     | The number of dependents (e.g., children) the customer has. |
| `estimated_salary`   | The customer's estimated monthly income in Naira. |

- `telecom_usage.csv` contains customer usage behaviour:

| Variable      | Description                                                  |
|---------------|--------------------------------------------------------------|
| `customer_id` | Unique identifier for each customer.                         |
| `calls_made`  | The number of calls made by the customer in the last 30 days. |
| `sms_sent`    | The number of SMS messages sent by the customer in the last 30 days. |
| `data_used`   | The amount of mobile data used by the customer (in MB).      |
| `churn`       | Binary variable: 1 = churned (left the provider), 0 = still active. |

> **Business context:** A churn prediction model that is deployed by a telecom company could be used to flag at-risk customers and trigger proactive retention campaigns — such as personalised data bonuses, loyalty rewards, or targeted customer care calls. The accuracy of the model directly affects how efficiently the business can allocate its retention budget.


In [2]:
# Import libraries and methods/functions
# Hint: You will need pandas, scikit-learn (for preprocessing, model selection, and classifiers),
# and optionally numpy.

---

# Task Overview

**Core Question:** Does Logistic Regression or a Random Forest Classifier produce a higher accuracy score in predicting customer churn for a Nigerian telecom provider?

---

## Task 1 — Load, Merge, and Explore

Load `telecom_demographics.csv` and `telecom_usage.csv` into separate DataFrames, then merge them into a single DataFrame named `churn_df` using the shared `customer_id` column.

After merging:
- Calculate the **proportion of customers who have churned** (i.e., the churn rate). Think about what this tells you from a business standpoint — is it a balanced or imbalanced dataset?
- Identify all **categorical variables** in `churn_df`. These will need to be handled before modelling.

> **Real-world note:** Churn datasets are often imbalanced — far more customers stay than leave. Be mindful of this as you interpret accuracy scores later.


Task 1: Load and Explore the Data
This stage involves importing the required liberies and loading the two datasets used for the analysis.
pandas was used for data manipulation and merging.
sklearn was used for preprocessing, model building, and evaluation.
The datasets include customer demographic information and telecome usage behaviour, with churn serving as the target variable.
The .info() output shows that both datasets contain 6500 records with no missing values, while .head() comfirms that the data loaded correctly and provide an initial view of the dataset structure and values.

In [ ]:
# Task 1: Load, merge, and explore
# Import libraries

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
# load the two dataset
demo_df = pd.read_csv(r"C:\Users\HP ELITEBOOK\Downloads\telecom_demographics_nigeria.csv")
use_df = pd.read_csv(r"C:\Users\HP ELITEBOOK\Downloads\telecom_usage.csv")
# check the data types of the dataset
demo_df.info()
use_df.info()
# print 5 rows of each dataset
print("Demographics Data:", demo_df
.head())
print("\nUsage Data:", use_df.head())



<class 'pandas.DataFrame'>
RangeIndex: 6500 entries, 0 to 6499
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   customer_id         6500 non-null   int64
 1   telecom_partner     6500 non-null   str  
 2   gender              6500 non-null   str  
 3   age                 6500 non-null   int64
 4   state               6500 non-null   str  
 5   city                6500 non-null   str  
 6   pincode             6500 non-null   int64
 7   registration_event  6500 non-null   str  
 8   num_dependents      6500 non-null   int64
 9   estimated_salary    6500 non-null   int64
dtypes: int64(5), str(5)
memory usage: 507.9 KB
<class 'pandas.DataFrame'>
RangeIndex: 6500 entries, 0 to 6499
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  6500 non-null   int64
 1   calls_made   6500 non-null   int64
 2   sms_sent     6500 non-null   int

In [7]:
# check for missing value in customer demographics datasets
demo_df.isnull().sum()

customer_id           0
telecom_partner       0
gender                0
age                   0
state                 0
city                  0
pincode               0
registration_event    0
num_dependents        0
estimated_salary      0
dtype: int64

In [8]:
# check for missing value in customer usage datasets
use_df.isnull().sum()

customer_id    0
calls_made     0
sms_sent       0
data_used      0
churn          0
dtype: int64

Exploying Unique Values in EAch Column
Examining unique values helps identify categorical variables, detect data quality issues, and determine which features require preprocessing.
Key findings include:
    Categorical columns such as telecom_partner, gender, and state require encoding.
    indentifier columns like customer_id and pincode will be removed.
    invalid negative values were found in calls_made, data_used and sms_sent and will be corrected.
    Numerical features are generally valid, although estimated_salary will require scaling.
These observations guild the data cleaning and preprocessing steps before model training.

In [ ]:
# Loop through the column in the customer demographics dataset to obtain unique values
for col in demo_df.columns:
    # Print the column name
    print(f"\nColumn: {col}")
    # Print the unique values in the column
    print(demo_df[col].unique())


Column: customer_id
[ 15169 149207 148119 ...  40413  64961  60427]

Column: telecom_partner
<StringArray>
['Airtel Nigeria', 'MTN Nigeria', 'Glo', '9mobile']
Length: 4, dtype: str

Column: gender
<StringArray>
['F', 'M']
Length: 2, dtype: str

Column: age
[26 74 54 29 45 69 34 55 50 38 27 20 47 62 31 73 24 23 63 70 51 39 42 66
 37 44 43 65 60 56 53 36 57 35 21 19 41 68 48 71 58 33 30 64 22 52 59 46
 49 18 72 28 67 25 32 61 40]

Column: state
<StringArray>
[      'Lagos',        'Kano',      'Rivers',         'Oyo',      'Kaduna',
     'Anambra',       'Enugu',       'Delta',        'Ogun',         'Edo',
     'Katsina',       'Borno',         'Imo',     'Plateau',       'Benue',
   'Akwa Ibom',       'Niger',        'Ondo',       'Kwara',     'Adamawa',
 'Cross River',        'Osun',       'Ekiti',      'Sokoto',      'Bauchi',
       'Kebbi',      'Taraba',   'FCT Abuja']
Length: 28, dtype: str

Column: city
<StringArray>
['Abuja', 'Port Harcourt', 'Kano', 'Ibadan', 'Enugu', 'Lagos'

In [ ]:
# Loop through columns of customer  dataset
for col in use_df.columns:
    # Print the column name 
    print(f"\nColumn: {col}")
    # Print the unique values in the column
    print(use_df[col].unique())


Column: customer_id
[ 15169 149207 148119 ...  40413  64961  60427]

Column: calls_made
[ 75  35  70  95  66  65  16  45  -4  22  88  80  26  49  12  90  76  51
  42  30   2  28  33  44  63  93  83  61  52  92  82  50   6  25  96  21
  20  15   0  56  67  62 100  55  69  86  32  24  34   5  38  27  89  64
  40  14  79  29  59  78  17  19  46   4 101  60   9  18  73  57  68 105
  87  39 102   7  -1  47  85  -3  72  74  10  13  36  91  77  53   3  37
  48  84  81  11  43  71  99  41  97  -5  -2   8  94  98  31  54   1 106
  23  -9 103 -10  -6  -8  58 104  -7 107 108]

Column: sms_sent
[21 38 47 32 23 18 17 -2 51 36  4 16 39 42 19  1  2 14 25 40 12 41  7 43
 45  0 34 15 11 26 22 35 13  8  9 33  5 -3 20 24 31 37 -1 28 48 30 50  3
 29 27  6 49 44 52 46 53 10 -4 -5]

Column: data_used
[4532  723 4688 ... 5000   65 6835]

Column: churn
[1 0]


Checking and Fixing Negative Vlues
Negative values found are invalid and indicate data quality issues. Around 2.6-2.7% of records contained negative values across these columns.
Further analysis showed that customers with negative values had churn rates similar to the overall dataset, suggesting that these values do not carry meaningful predictive information.
Therefore, all negative values were replaced with 0 using .clip(lower=0). This approach preserves valid customer records while correcting the invalid entries. After the correction, all three columns contained no negative values and were ready for modelling. 

In [4]:
# Check for negative values in the columns: calls_made, sms_sent, data_used and print the summary statistics 
def check_negative_values(df, column, sample_size=10):
    columns = ["calls_made", "sms_sent", "data_used"]
    print("=== Negative Value Investigation ===\n")
    for col in columns:
        neg_values = df[df[col] < 0]
        print(f"Column: {col}")
        print(f" Found {len(neg_values)} rows with negative values.")
        if not neg_values.empty:
            print(" Summary statistics for negative values:")
            print(neg_values[col].describe())
            print(f"\n Sample of offending rows (up to {sample_size}):")
            print(neg_values.head(sample_size))
        else:
            print(" No negative values found.")
        print("\n" + "="*50 + "\n")
check_negative_values(use_df, ["calls_made", "sms_sent", "data_used"])



=== Negative Value Investigation ===

Column: calls_made
 Found 178 rows with negative values.
 Summary statistics for negative values:


count    178.000000
mean      -3.932584
std        2.519070
min      -10.000000
25%       -5.750000
50%       -3.000000
75%       -2.000000
max       -1.000000
Name: calls_made, dtype: float64

 Sample of offending rows (up to 10):
     customer_id  calls_made  sms_sent  data_used  churn
8          48459          -4        51       8292      1
129        74040          -1         7       4227      1
139       112432          -3        15       7533      1
177          191          -3        -1       3838      1
221        10407          -3        51       8398      1
276        82005          -4         6       1930      1
278       159927          -5        12       5517      1
283       135015          -2        22       5457      1
327       220138          -5        25       5867      1
345       181274          -1        25       8404      1


Column: sms_sent
 Found 167 rows with negative values.
 Summary statistics for negative values:
count    167.000000
mean      -2.233533
std

In [5]:
# investigate relationship between negative values and churn
print("=" * 60)
print("NEGATIVE VALUES vs CUSTOMER CHURN")
print("=" * 60)

# Create flags for negative values
use_df["neg_calls"] = use_df["calls_made"] < 0
use_df["neg_sms"]   = use_df["sms_sent"] < 0
use_df["neg_data"]  = use_df["data_used"] < 0

# Cross-tabulate with churn
print("\n--- Calls Made (negativity vs Customer Churn) ---")
print(pd.crosstab(use_df["neg_calls"], use_df["churn"],
                  rownames=["Has Negative Calls"],
                  colnames=["Churned"],
                  margins=True))

print("\n--- SMS Sent (negativity vs Customer Churn) ---")
print(pd.crosstab(use_df["neg_sms"], use_df["churn"],
                  rownames=["Has Negative SMS"],
                  colnames=["Churned"],
                  margins=True))

print("\n--- Data Used (negativity vs Customer Churn) ---")
print(pd.crosstab(use_df["neg_data"], use_df["churn"],
                  rownames=["Has Negative Data"],
                  colnames=["Churned"],
                  margins=True))

# Calculate churn rate for negative vs non-negative groups
print("\n--- Churn Rate by Calls Made ---")
print(use_df.groupby("neg_calls")["churn"].mean().rename({False: "Non-Negative", True: "Negative"}))

print("\n--- Churn Rate by SMS Sent ---")
print(use_df.groupby("neg_sms")["churn"].mean().rename({False: "Non-Negative", True: "Negative"}))

print("\n--- Churn Rate by Data Used ---")
print(use_df.groupby("neg_data")["churn"].mean().rename({False: "Non-Negative", True: "Negative"}))

NEGATIVE VALUES vs CUSTOMER CHURN

--- Calls Made (negativity vs Customer Churn) ---
Churned                0     1   All
Has Negative Calls                  
False               5055  1267  6322
True                 142    36   178
All                 5197  1303  6500

--- SMS Sent (negativity vs Customer Churn) ---
Churned              0     1   All
Has Negative SMS                  
False             5067  1266  6333
True               130    37   167
All               5197  1303  6500

--- Data Used (negativity vs Customer Churn) ---
Churned               0     1   All
Has Negative Data                  
False              5072  1262  6334
True                125    41   166
All                5197  1303  6500

--- Churn Rate by Calls Made ---
neg_calls
Non-Negative    0.200411
Negative        0.202247
Name: churn, dtype: float64

--- Churn Rate by SMS Sent ---
neg_sms
Non-Negative    0.199905
Negative        0.221557
Name: churn, dtype: float64

--- Churn Rate by Data Used ---
neg

In [6]:
# Remove investigation flag  (there's no significant relationship)
use_df.drop(columns=["neg_calls", "neg_sms", "neg_data"], inplace=True, errors="ignore")

# Replace negative values with 0
use_df["calls_made"] = use_df["calls_made"].clip(lower=0)
use_df["sms_sent"]   = use_df["sms_sent"].clip(lower=0)
use_df["data_used"]  = use_df["data_used"].clip(lower=0)

# Print result for confirmation
print("=== Final Confirmation ===")
print(f"Remaining negative values:")
print(f"calls_made : {(use_df['calls_made'] < 0).sum()}")
print(f"sms_sent   : {(use_df['sms_sent'] < 0).sum()}")
print(f"data_used  : {(use_df['data_used'] < 0).sum()}")
print("\nFinal  stats:")
print(use_df[["calls_made", "sms_sent", "data_used"]].describe())

=== Final Confirmation ===
Remaining negative values:
calls_made : 0
sms_sent   : 0
data_used  : 0

Final  stats:
        calls_made     sms_sent     data_used
count  6500.000000  6500.000000   6500.000000
mean     49.897231    24.315231   5009.393538
std      29.608445    14.549686   2925.485632
min       0.000000     0.000000      0.000000
25%      25.000000    12.000000   2493.750000
50%      50.000000    25.000000   4975.500000
75%      75.000000    37.000000   7504.250000
max     108.000000    53.000000  10919.000000


Merge, Explore, and Understand the Data
The cleaned demographic and usage datasets were merged using an inner join on customer_id, resulting in a dataset of 6500 customers and 14 features with no missing values.
The churn distribution shows that 20.05% of customers have churned, while 79.95% remain active, indicating a moderately imbalance dataset. As a result, evaluation metrics beyond accuracy, such as recall, are important during model assessment.

In [9]:
# Merge Customer Demographics and Usage datasets on 'customer_id'
churn_df = pd.merge(demo_df, use_df, on="customer_id", how="inner")
# Quick check of the merged DataFrame
print("=== Merged Summary ===")
print(f"Shape: {churn_df.shape}")
print(f"Churn rate: {churn_df['churn'].mean():.2%}")
print(f"Missing values: {churn_df.isnull().sum().sum()}")
# Preview
print("\nSample Rows:")
print(churn_df.head())
print("\nStats:")
print(churn_df.describe())

=== Merged Summary ===
Shape: (6500, 14)
Churn rate: 20.05%
Missing values: 0

Sample Rows:
   customer_id telecom_partner gender  age   state           city  pincode  \
0        15169  Airtel Nigeria      F   26   Lagos          Abuja   667173   
1       149207  Airtel Nigeria      F   74    Kano  Port Harcourt   313997   
2       148119  Airtel Nigeria      F   54  Rivers           Kano   549925   
3       187288     MTN Nigeria      M   29     Oyo  Port Harcourt   230636   
4        14016             Glo      M   45  Kaduna         Ibadan   188036   

  registration_event  num_dependents  estimated_salary  calls_made  sms_sent  \
0         2020-03-16               4            648530          75        21   
1         2022-01-16               0            506057          35        38   
2         2022-01-11               2            562102          70        47   
3         2022-07-26               3            202972          95        32   
4         2020-03-11               4   

In [10]:
#  Churn Overview
churn_rate = churn_df["churn"].mean()
print("=== Churn Summary ===")
print(f"Churn Rate: {churn_rate:.2%}")

# Identify categorical variables
cat_cols = churn_df.select_dtypes(include=["string", "object"]).columns.tolist()
print(f"Categorical Variables: {cat_cols}")

# Customer counts
total   = len(churn_df)
churned           = churn_df["churn"].sum()
active            = total - churned

print(f"\nTotal Customers : {total:,}")
print(f"Churned         : {churned:,}  ({churn_rate:.2%})")
print(f"Active          : {active:,}  ({1 - churn_rate:.2%})")
# Dataset balance assessment
balance = "Moderately Imbalanced" if 0.15 <= churn_rate <= 0.40 else "Highly Imbalanced"
print(f"\nDataset balance : {balance}")

=== Churn Summary ===
Churn Rate: 20.05%
Categorical Variables: ['telecom_partner', 'gender', 'state', 'city', 'registration_event']

Total Customers : 6,500
Churned         : 1,303  (20.05%)
Active          : 5,197  (79.95%)

Dataset balance : Moderately Imbalanced


---

## Task 2 — Preprocess Features

Prepare the data for machine learning by doing the following:

1. **Encode categorical features** — Convert all categorical columns (e.g., `telecom_partner`, `gender`, `state`) into numeric format using one-hot encoding or label encoding.
2. **Select features** — Drop columns that would not be useful for prediction (e.g., `customer_id`, `city`, `pincode`, `registration_event`).
3. **Scale numeric features** — Apply feature scaling to the numerical columns. Store the result in a DataFrame called `features_scaled`.
4. **Define your target** — Isolate the `churn` column as your target variable `y`.

> **Why does scaling matter?** Logistic Regression is sensitive to feature magnitude — a salary in the millions of Naira and an age in the tens would otherwise be weighted very differently. Random Forest is less affected, but scaling is still good practice for fair comparison.


Task 2:  Preprocess Features
Categorical variables were encoded, irrelevant columns were removed, and numerical features were scaled to prepare the data modelling.
After preprocessing, the target variable (y) and feature matrix (X) were separated, resulting in 37 numeric features with no missing values. The final scaled dataset is clean and ready for model training.

In [11]:
# Task 2: Encode, select features, scale, and define target

# Encode categorical columns 
churn_df["gender"] = churn_df["gender"].map({"M": 1, "F": 0})
churn_df = pd.get_dummies(churn_df,
                          columns=["telecom_partner", "state"],
                          drop_first=True)
# Print the shape after encoding
print(f"After encoding : {churn_df.shape}")

# Drop non-model columns 
drop_cols = ["customer_id", "city", "pincode", "registration_event"]
churn_df = churn_df.drop(columns=drop_cols)
# Print the shape after dropping
print(f"After dropping : {churn_df.shape}")

#  target and features matrix
y = churn_df["churn"]
X = churn_df.drop(columns=["churn"])
X = X.astype({col: int for col in X.select_dtypes(include="bool").columns})
print(f"\ny shape: {y.shape}, Churn rate: {y.mean():.2%}")
print(f"X shape: {X.shape}")

# Scale numeric features
num_cols = ["age", "num_dependents", "estimated_salary", "calls_made", "sms_sent", "data_used"]
# instantiate StandardScaler
scaler = StandardScaler()
# Create copy of X to hold scaled features
X_scaled = X.copy()
# fit the numeric columns
X_scaled[num_cols] = scaler.fit_transform(X[num_cols])
features_scaled = X_scaled

# Preview the scaled features
print(f"\nfeatures_scaled: {features_scaled.shape}")
print(f"Missing in X: {features_scaled.isnull().sum().sum()}, Missing in y: {y.isnull().sum()}")
print(f"Means (~0): features_scaled[num_cols].mean().round(4)")
print(f"Stds (~1): features_scaled[num_cols].std().round(4)")
print(f"\nSample:")
print(features_scaled.head(3))

After encoding : (6500, 42)
After dropping : (6500, 38)

y shape: (6500,), Churn rate: 20.05%
X shape: (6500, 37)

features_scaled: (6500, 37)
Missing in X: 0, Missing in y: 0
Means (~0): features_scaled[num_cols].mean().round(4)
Stds (~1): features_scaled[num_cols].std().round(4)

Sample:
   gender       age  num_dependents  estimated_salary  calls_made  sms_sent  \
0       0 -1.222970        1.436539          0.011980    0.847890 -0.227873   
1       0  1.696304       -1.411346         -0.428424   -0.503180  0.940627   
2       0  0.479940        0.012596         -0.255181    0.679006  1.559244   

   data_used  telecom_partner_Airtel Nigeria  telecom_partner_Glo  \
0  -0.163197                               1                    0   
1  -1.465303                               1                    0   
2  -0.109868                               1                    0   

   telecom_partner_MTN Nigeria  ...  state_Lagos  state_Niger  state_Ogun  \
0                            0  ...   

---

## Task 3 — Train Models

Split your data into training and testing sets, then train two classification models:

1. **Split the data** into `X_train`, `X_test`, `y_train`, and `y_test` using an **80/20 split**, with `random_state=42` for reproducibility.
2. **Train a Logistic Regression model** — Use `random_state=42`. Store its predictions on the test set in `logreg_pred`.
3. **Train a Random Forest Classifier** — Use `random_state=42`. Store its predictions on the test set in `rf_pred`.

> **Real-world note:** In production, models are re-trained periodically on fresh data. The 80/20 split simulates this: training on historical data and testing on "unseen" new customers. Setting a random state ensures your results are reproducible — an important requirement in professional data science workflows.


Train-Test Split and Model Training:
The dataset was split into 80% training data and 20% testing data to evaluate model performance on unseen data. Two models, Logistic Regression and Random Forest, were trained using class weights to address the imbalance in the churn classes.
Both models generated predictions on the test set and are ready for performance evaluation.

In [12]:
# Task 3: Train-test split and model training
# Split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(features_scaled,
                                                    y, test_size=0.2, random_state=42)
# Preview data on training and testing
print("=== Split Summary ===")
print(f"Train: {X_train.shape[0]} rows ({X_train.shape[0]/len(features_scaled):.0%})")
print(f"Test: {X_test.shape[0]} rows ({X_test.shape[0]/len(features_scaled):.0%})")
print(f"Train churn rate : {y_train.mean():.2%}")
print(f"Test churn rate: {y_test.mean():.2%}")

#  Logistic Regression 
logreg_model = LogisticRegression(random_state=42, class_weight={0: 1, 1: 3}, max_iter=1000)
# Fit the logreg_model
logreg_model.fit(X_train, y_train)
# Predict the logreg_ model
logreg_pred  = logreg_model.predict(X_test)


# Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42, class_weight={0: 1, 1: 4}, n_estimators=200, min_samples_leaf=5)
# Fit the rf_model
rf_model.fit(X_train, y_train)
# Predict the rf_model
rf_pred  = rf_model.predict(X_test)


# Print the shape and sample of predictions
print("\n=== Prediction Summary ===")
print(f"\nlogreg_pred shape: {logreg_pred.shape}, Sample logreg_pred: {logreg_pred[:10]}")
print(f"rf_pred shape: {rf_pred.shape}, Sample rf_pred: {rf_pred[:10]}")

=== Split Summary ===
Train: 5200 rows (80%)
Test: 1300 rows (20%)
Train churn rate : 19.81%
Test churn rate: 21.00%

=== Prediction Summary ===

logreg_pred shape: (1300,), Sample logreg_pred: [0 0 0 0 0 0 1 0 1 0]
rf_pred shape: (1300,), Sample rf_pred: [1 1 0 0 0 0 0 0 1 0]


---

## Task 4 — Evaluate and Compare Models

Assess both models on the test data:

1. **Calculate accuracy scores** for both `logreg_pred` and `rf_pred` against `y_test`.
2. **Print a classification report** for each model to understand precision, recall, and F1-score — these give a fuller picture than accuracy alone.
3. **Assign the name of the better-performing model** (by accuracy) to a variable called `higher_accuracy`. Use exactly one of these strings: `"LogisticRegression"` or `"RandomForest"`.

> **Reflection question (no code needed):** If you were advising the telecom company on which model to deploy, would accuracy alone be enough to make the decision? Consider: what is the cost of a *false negative* (predicting a customer won't churn when they actually will)?


Evaluate and Compare Models: 
Both models were evaluated using accuracy and classification metrics. While Logistic Regression achieved higher accuracy, Random Forest performed better at identifying churned customers. 

In [13]:
# Task 4: Model evaluation
# Accuracy scores 
logreg_acc = accuracy_score(y_test, logreg_pred)
rf_acc     = accuracy_score(y_test, rf_pred)

# Print accuracy scores
print(f"Logistic Regression Accuracy: {logreg_acc:.4f}")
print(f"Random Forest Accuracy      : {rf_acc:.4f}")

# Classification reports
print("\nLogistic Regression  Report")
print(classification_report(y_test, logreg_pred,
                            target_names=["Active", "Churned"],
                            zero_division=0))

print("\nRandom Forest Report")
print(classification_report(y_test, rf_pred,
                            target_names=["Active", "Churned"],
                            zero_division=0))

# Assign better-performing model by accuracy
higher_accuracy = "RandomForest" if rf_acc > logreg_acc else "LogisticRegression"
print(f"Higher accuracy model : {higher_accuracy}")

Logistic Regression Accuracy: 0.7323
Random Forest Accuracy      : 0.6854

Logistic Regression  Report
              precision    recall  f1-score   support

      Active       0.79      0.91      0.84      1027
     Churned       0.17      0.07      0.10       273

    accuracy                           0.73      1300
   macro avg       0.48      0.49      0.47      1300
weighted avg       0.66      0.73      0.69      1300


Random Forest Report
              precision    recall  f1-score   support

      Active       0.79      0.82      0.81      1027
     Churned       0.20      0.16      0.18       273

    accuracy                           0.69      1300
   macro avg       0.49      0.49      0.49      1300
weighted avg       0.66      0.69      0.67      1300

Higher accuracy model : LogisticRegression


When advising a telecom company on which churn prediction model to deploy, relying solely on accuracy can lead to the wrong decision. Although Logistic regression achieved a higher overall accuracy of 73%, it performed poorly in identifying customers who are likely to leave, with a recal score of just 7%. this mean the model failed to detect most of the actual churners. 
From a business perspective, these missed churners are particularly costly. every customer who leaves without being identified represents a lost opportunity for the company to apply retention strategies, maintain customer relationships, and protect revenue.
On the other hand, the Random Forest model, despite having a lower overall accuracy of 68.5%, performed slightly better in detecting churners, achieving a recall of 16%. While this level of recall is still relatively low, it highlights an important issue in churn prediction that accuracy alone can be misleading, especially when working with imbalanced datasets. In such cases, the ability to correctly identify customers at risk of leaving is often more valuable than accurately classifying customers who are likely to stay.
Therefore, the company should avoid making deployment decisions based solely on accuracy. Greater emphasis should be placed on improving the recall of churning customers, even if this results in a slight reduction in overall accuracy. Approaches such as adjusting class weights, oversampling churn cases, or exploring bossting model may help achieve a better balance between overall performance and churn detection.
In conclusion, the preferred model should be the one that identifies the highest number of potential churners. Missing customers who are likely to leave can have a much greater finacial impact than incorrectly flagging a few loyal customers as being at risk.